# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art on Kaggle GPU

This notebook dynamically detects the Kaggle GPU runtime, configures headless WebGL via Node 20 LTS + Puppeteer Chrome + Xvfb, loads trained LoRA policies, and runs reinforcement learning training cycles.

### ⚡ Key Capabilities:
1. **Full Frame Canvas Capture**: Waits for `draw()` to finish execution so watercolor strokes never render blank.
2. **Procedural Image Fallbacks**: Automatically shims `loadImage` so texture hallucinations never stall rendering.
3. **Process Recycling**: Automatically flushes stale processes on port 3000 to ensure fresh code execution.
4. **Official Puppeteer Chrome**: Native Chrome binary installation bypassing Ubuntu's non-functional snap stub.
5. **Zero Conflicts**: Neutralizes Kaggle's incompatible `torchao` to prevent PEFT import errors.
6. **Tesla T4 Auto-Tuning**: Automatically selects `Qwen2.5-Coder-1.5B-Instruct` for 15.6GB VRAM.
7. **Instant Model Showcase**: Automatically downloads and evaluates `pernavjain/paint-code/pyTorch/default`.
8. **GRPO Cyclic Training**: Optimizes p5.js syntax and pixel-space visual richness with live dashboard tracking.
9. **One-Click Export**: Bundles all renders and checkpoints into a downloadable ZIP.

In [ ]:
# Cell 1: Environment & GPU Auto-Probing
import os, sys, platform, torch, psutil

print('=' * 60)
print('   PAINT-CODE-RL: KAGGLE RUNTIME INITIALIZATION')
print('=' * 60)
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  [GPU {i}] {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)')
else:
    print('  [WARN] Running on CPU. For fast GPU acceleration, enable GPU in Settings.')

ram = psutil.virtual_memory()
print(f'RAM: {ram.total / 1e9:.1f} GB Total / {ram.available / 1e9:.1f} GB Available')
print('=' * 60)

In [ ]:
# Cell 2: Install Node.js 20 LTS, Xvfb, and Chrome Runtime Libraries
# 1. Precompiled Node 20 LTS binary into /usr/local
!curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1

# 2. Remove any Ubuntu snap redirection stubs
!rm -f /usr/bin/chromium-browser /usr/bin/chromium

# 3. Linux X11/GL shared libraries required by headless Chrome
!apt-get update -qq && apt-get install -y -qq \
    libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
    libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 \
    xvfb > /dev/null 2>&1

import os
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"
!/usr/local/bin/node -v && /usr/local/bin/npm -v

In [ ]:
# Cell 3: Clone Repository and Hard-Reset to Latest Commit (Idempotent & Non-Blocking)
import os, shutil
os.environ["GIT_TERMINAL_PROMPT"] = "0"

if not os.path.exists("/kaggle/working/paint-code-rl/.git"):
    if os.path.exists("/kaggle/working/paint-code-rl"):
        shutil.rmtree("/kaggle/working/paint-code-rl", ignore_errors=True)
    !git clone --depth 1 https://github.com/harshitthek/paint-code-rl.git /kaggle/working/paint-code-rl

os.chdir("/kaggle/working/paint-code-rl")
!git fetch origin feat/visual-rl-and-cyclic-training
!git reset --hard origin/feat/visual-rl-and-cyclic-training
print("Working directory:", os.getcwd())

In [ ]:
# Cell 4: Install Python & Node.js Dependencies (with Puppeteer Chrome)
import os
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

# Neutralize Kaggle's incompatible pre-installed torchao 0.10.0 to prevent PEFT crash
!pip uninstall -y torchao > /dev/null 2>&1
!pip install -q trl==0.15.1 transformers==4.49.0 peft datasets accelerate pydantic safetensors Pillow pyyaml psutil requests kagglehub

os.chdir("/kaggle/working/paint-code-rl/renderer")
!/usr/local/bin/npm install --no-audit --no-fund
!npx puppeteer browsers install chrome --install-deps
os.chdir("/kaggle/working/paint-code-rl")
print("[OK] Dependencies installed successfully!")

In [ ]:
# Cell 5: Launch Headless WebGL Rendering Daemon with Xvfb
import subprocess, time, requests, os
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

# Kill any stale node daemon holding port 3000
!pkill -9 -f "node.*server.js" || true
!rm -f /usr/bin/chromium-browser /usr/bin/chromium
time.sleep(1)

# Start renderer daemon under xvfb virtual frame buffer
proc = subprocess.Popen(["xvfb-run", "-a", "/usr/local/bin/node", "renderer/server.js"])
time.sleep(4)

# Verify renderer health
for attempt in range(5):
    try:
        health = requests.get("http://127.0.0.1:3000/health", timeout=3).json()
        print("[OK] Renderer daemon is live:", health)
        break
    except Exception:
        time.sleep(2)
else:
    print("[WARN] Renderer daemon did not respond immediately, continuing...")

In [ ]:
# Cell 6: Generate Artwork Using Your Uploaded KaggleHub Checkpoint
# Automatically pulls pernavjain/paint-code/pyTorch/default and renders high-res artworks
!python scripts/generate_and_render.py --kagglehub pernavjain/paint-code/pyTorch/default --output-dir artifacts/renders --temperature 0.4 --max-new-tokens 550

import glob
from IPython.display import display, Image

renders = sorted(glob.glob("artifacts/renders/render_*.png"))
print(f"Total artworks generated: {len(renders)}")
for r in renders:
    print(f"File: {r}")
    display(Image(filename=r, width=400))

In [ ]:
# Cell 7: Run GRPO Cyclic Training on GPU with Auto Hardware Saturation
import os
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

# Run 25 training steps with auto hardware saturation and live dashboard
!python scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 25 --unattended --max --dashboard

In [ ]:
# Cell 8: View Live Dashboard Inline
from IPython.display import display, HTML
if os.path.exists("artifacts/dashboard.html"):
    with open("artifacts/dashboard.html", "r", encoding="utf-8") as f:
        html_code = f.read()
    display(HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;")}" width="100%" height="600px" frameborder="0"></iframe>'))

In [ ]:
# Cell 9: Package All Outputs into Downloadable ZIP
# After running this, download 'paint_rl_artifacts.zip' from Kaggle's right-hand Output panel!
!python scripts/package_artifacts.py --output-dir /kaggle/working
print("\n[DONE] Check the right-hand panel under 'Output' to download your ZIP file!")